# P2 · Fleet Maintenance Planning

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
INSPECTION, SHORTAGE, SURPLUS = 10.0, 0.20, 0.02
SENSORS = [f"s{i:03d}" for i in range(170)]
SCHEMA = ["id", "flag", "budget"]

def find_data():
    required = {"train.csv", "test.csv", "sample_submission.csv"}
    kaggle = Path("/kaggle/input")
    candidates = [Path.cwd(), *(p.parent for p in sorted(kaggle.rglob("train.csv")))]
    matches = []
    for folder in dict.fromkeys(candidates):
        if all((folder / name).is_file() for name in required):
            test_cols = set(pd.read_csv(folder / "test.csv", nrows=0).columns)
            train_cols = set(pd.read_csv(folder / "train.csv", nrows=0).columns)
            if {"id", *SENSORS} <= test_cols and {"failure", "repair_cost", *SENSORS} <= train_cols:
                matches.append(folder)
    if len(matches) != 1:
        raise ValueError(f"Expected one P2 dataset. Matching directories: {matches}")
    return matches[0]

data = find_data()
train = pd.read_csv(data / "train.csv", dtype={"id": str})
test = pd.read_csv(data / "test.csv", dtype={"id": str})
sample = pd.read_csv(data / "sample_submission.csv", dtype={"id": str})
assert len(train) == 56_000 and len(test) == len(sample) == 20_000
assert list(sample) == SCHEMA
assert test.id.is_unique and sample.id.is_unique and set(test.id) == set(sample.id)
assert train.failure.isin([0, 1]).all()
failure = train.failure.to_numpy(dtype=int)
repair = train.repair_cost.fillna(0).to_numpy()
assert np.isfinite(repair).all() and np.array_equal(repair > 0, failure == 1)
print(f"{len(train):,} training vehicles, {failure.sum():,} recorded failures")

56,000 training vehicles, 397 recorded failures


## Sensor selection

Drop sensors with fewer than two distinct values in either depot, or with a KS statistic or missing-rate difference of at least 0.10.

In [2]:
shift = {}
for name in SENSORS:
    old, new = train[name].dropna(), test[name].dropna()
    if min(old.nunique(), new.nunique()) < 2:
        shift[name] = 1.0
    else:
        distribution_shift = ks_2samp(old, new, method="asymp").statistic
        missingness_shift = abs(len(old) / len(train) - len(new) / len(test))
        shift[name] = max(distribution_shift, missingness_shift)

stable = [name for name in SENSORS if shift[name] < 0.10]

frequently_missing = train[stable].isna().mean().to_numpy() >= 0.20

def features(frame):
    readings = frame[stable].to_numpy(dtype="float32")
    missing = np.isnan(readings)
    counts = missing.sum(axis=1, dtype="float32")
    group_counts = missing[:, frequently_missing].sum(axis=1, dtype="float32")
    return np.column_stack((readings, counts, group_counts))

X, X_test = features(train), features(test)
print(f"Kept {len(stable)} sensors. Excluded:", sorted(set(SENSORS) - set(stable)))

Kept 156 sensors. Excluded: ['s047', 's054', 's055', 's056', 's057', 's058', 's059', 's060', 's061', 's062', 's063', 's120', 's122', 's146']


## Failure risk and repair cost

Fit two failure classifiers, a repair-cost-weighted classifier, and a log-cost regressor on failed vehicles.

In [3]:
def fit_predict(fit_idx, query, seed):
    x, y, c = X[fit_idx], failure[fit_idx], repair[fit_idx]
    positive = y == 1
    cls_options = dict(learning_rate=0.075, max_iter=210, max_bins=127,
                       early_stopping=False, random_state=seed)
    probabilities = []
    for leaves, min_leaf in ((7, 25), (15, 40)):
        clf = HistGradientBoostingClassifier(
            **cls_options, max_leaf_nodes=leaves, min_samples_leaf=min_leaf,
            l2_regularization=3.0, max_features=0.85)
        clf.fit(x, y)
        probabilities.append(clf.predict_proba(query)[:, 1])
    p = np.mean(probabilities, axis=0)

    mean_cost = c[positive].mean()
    weights = np.where(positive, c / mean_cost, 1.0)
    weighted = HistGradientBoostingClassifier(
        **cls_options, max_leaf_nodes=7, min_samples_leaf=35,
        l2_regularization=5.0, max_features=0.85)
    weighted.fit(x, y, sample_weight=weights)
    p_weighted = np.clip(weighted.predict_proba(query)[:, 1], 0, 0.985)
    expected_cost = mean_cost * p_weighted / (1 - p_weighted) * (1 - p)

    severity = HistGradientBoostingRegressor(
        max_leaf_nodes=5, min_samples_leaf=20, max_iter=65,
        learning_rate=0.06, l2_regularization=4.0, early_stopping=False,
        random_state=seed)
    severity.fit(x[positive], np.log(c[positive]))
    return np.column_stack((p, expected_cost, severity.predict(query)))


def cross_fit(indices, query, seed, n_splits):
    oof = np.empty((len(indices), 3))
    predictions = []
    split_y = failure[indices]
    splitter = StratifiedKFold(n_splits, shuffle=True, random_state=seed)
    for fold, (fit, valid) in enumerate(splitter.split(indices, split_y), start=1):
        combined = np.vstack((X[indices[valid]], query))
        output = fit_predict(indices[fit], combined, seed + fold)
        oof[valid] = output[:len(valid)]
        predictions.append(output[len(valid):])
        print(f"Cross-fit fold {fold}/{n_splits}", flush=True)
    return oof, np.mean(predictions, axis=0)

## Inspection and parts budget

Budget at the 10/11 quantile of estimated repair demand, including zero demand for healthy vehicles. Inspect when the expected cost is lower than skipping.

In [4]:
def repair_distribution(prediction, reference, reference_cost, conditional):
    positive = reference_cost > 0
    observed = reference_cost[positive]
    if conditional:
        residuals = np.exp(np.log(observed) - reference[positive, 2])
        scale = np.exp(prediction[:, 2])
    else:
        residuals = observed
        scale = np.ones(len(prediction))
    values = np.sort(residuals)
    cumulative = np.r_[0.0, np.cumsum(values)]
    return scale, values, cumulative


def decisions(prediction, distribution, probability_scale, direct_share):
    scale, values, cumulative = distribution
    conditional_mean = scale * values.mean()
    direct_risk = prediction[:, 1] / conditional_mean
    p = np.clip(probability_scale * (
        (1 - direct_share) * prediction[:, 0] + direct_share * direct_risk
    ), 0.0, 1.0)
    fraction = 1 - SURPLUS / ((SHORTAGE + SURPLUS) * np.maximum(p, 1e-12))
    index = np.clip(np.ceil(fraction * len(values)).astype(int) - 1, 0, len(values) - 1)
    budget = np.where(fraction > 0, np.rint(scale * values[index]), 0.0)
    below = np.searchsorted(values, budget / scale, side="right")
    surplus = np.maximum(0.0, (budget * below - scale * cumulative[below]) / len(values))
    shortage = np.maximum(0.0, conditional_mean - budget + surplus)
    inspect_cost = INSPECTION + SHORTAGE * p * shortage + SURPLUS * ((1 - p) * budget + p * surplus)
    flag = p * conditional_mean > inspect_cost
    return flag.astype(int), np.where(flag, budget, 0).astype(int)


def total_cost(actual, flag, budget):
    inspected = INSPECTION + SHORTAGE * np.maximum(actual - budget, 0) + SURPLUS * np.maximum(budget - actual, 0)
    return np.where(flag == 1, inspected, actual).sum()


assert total_cost(np.array([400, 0, 1200, 0]), np.array([1, 1, 0, 0]),
                  np.array([300, 50, 0, 0])) == 1241

## Policy tuning and holdout

Tune on out-of-fold maintenance cost and evaluate once on a 25% holdout.

In [5]:
development, audit = train_test_split(np.arange(len(train)), test_size=0.25,
                                      stratify=failure, random_state=SEED)
dev_oof, audit_prediction = cross_fit(development, X[audit], SEED, n_splits=4)
dev_cost = repair[development]
dev_folds = list(StratifiedKFold(4, shuffle=True, random_state=SEED).split(
    development, failure[development]))

results = []
for conditional in (False, True):
    fold_distributions = [
        repair_distribution(dev_oof[valid], dev_oof[fit], dev_cost[fit], conditional)
        for fit, valid in dev_folds
    ]
    for probability_scale in (0.85, 1.0, 1.25, 1.6):
        for direct_share in (0.0, 0.35, 0.65):
            cost = 0.0
            for (_, valid), distribution in zip(dev_folds, fold_distributions):
                flag, budget = decisions(dev_oof[valid], distribution, probability_scale, direct_share)
                cost += total_cost(dev_cost[valid], flag, budget)
            results.append((cost, conditional, probability_scale, direct_share))
selection_cost, conditional, probability_scale, direct_share = min(results)

audit_dist = repair_distribution(audit_prediction, dev_oof, dev_cost, conditional)
audit_flag, audit_budget = decisions(audit_prediction, audit_dist, probability_scale, direct_share)
audit_cost = total_cost(repair[audit], audit_flag, audit_budget)
print(f"Policy: conditional severity={conditional}, risk scale={probability_scale}, direct share={direct_share}")
print(f"Development cost: {selection_cost:,.2f} on {len(development):,} vehicles")
print(f"Holdout probability average precision: {average_precision_score(failure[audit], audit_prediction[:, 0]):.4f}")
print(f"Holdout cost: {audit_cost:,.2f} on {len(audit):,} vehicles")
print(f"Holdout baseline, inspect nobody: {repair[audit].sum():,.2f}")
print(f"Holdout inspections: {audit_flag.sum():,}")

Cross-fit fold 1/4
Cross-fit fold 2/4
Cross-fit fold 3/4
Cross-fit fold 4/4
Policy: conditional severity=True, risk scale=1.25, direct share=0.0
Development cost: 41,025.76 on 42,000 vehicles
Holdout probability average precision: 0.5653
Holdout cost: 12,421.60 on 14,000 vehicles
Holdout baseline, inspect nobody: 52,194.00
Holdout inspections: 340


## Submission

Refit on all training data and write `/kaggle/working/submission.csv` on Kaggle.

In [6]:
all_oof, test_prediction = cross_fit(np.arange(len(train)), X_test, SEED + 1, n_splits=5)
test_dist = repair_distribution(test_prediction, all_oof, repair, conditional)
flag, budget = decisions(test_prediction, test_dist, probability_scale, direct_share)
predictions = pd.DataFrame({"id": test.id, "flag": flag, "budget": budget}).set_index("id")
submission = predictions.loc[sample.id].reset_index()

output_dir = Path("/kaggle/working")
if not output_dir.is_dir():
    output_dir = Path.cwd()
output = output_dir / "submission.csv"
submission.to_csv(output, index=False)
written = pd.read_csv(output, dtype={"id": str})
assert written.columns.tolist() == SCHEMA
assert len(written) == 20_000 and written.id.is_unique and written.id.equals(sample.id)
assert written.flag.isin([0, 1]).all()
assert np.isfinite(written.budget).all() and written.budget.ge(0).all()
assert written.loc[written.flag.eq(0), "budget"].eq(0).all()
print(f"Saved {len(written):,} rows to {output}")
print(f"Inspections: {written.flag.sum():,}. Positive budgets: {written.budget.gt(0).sum():,}.")
print(written.head().to_string(index=False))

Cross-fit fold 1/5
Cross-fit fold 2/5
Cross-fit fold 3/5
Cross-fit fold 4/5
Cross-fit fold 5/5
Saved 20,000 rows to /kaggle/working/submission.csv
Inspections: 942. Positive budgets: 724.
    id  flag  budget
t00001     0       0
t00002     0       0
t00003     0       0
t00004     0       0
t00005     0       0
